In [1]:
import math
import pandas as pd
import joblib 
from pathlib import Path
from sklearn.datasets import load_wine
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.preprocessing import StandardScaler


# Clientes com escalas diferentes 

In [ ]:
p1 = {"salary": 5000, "historico": 9} # bom pagador
p2 = {"salary": 5001, "historico": 1} # péssimo pagador

dx = p1["salary"] - p2["salary"]
dy = p1["historico"] - p2["historico"]
dist = math.sqrt(dx**2 + dy**2)

print(f"Distância sem normalização: {dist:.2f}")

Distância sem normalização: 8.06


# KNN StandarScalar - normalização
Sem a normalização, alguma linha com dado muito discrepante pode afetar tudo, por isso tem que normalizar

In [ ]:
df = pd.DataFrame([
    {"salary": 5000, "historico": 8, "aprovado": 1},
    {"salary": 1000, "historico": 2, "aprovado": 0},
    {"salary": 6000, "historico": 9, "aprovado": 1},
    {"salary": 2000, "historico": 3, "aprovado": 0},
    {"salary": 5000, "historico": 7, "aprovado": 1},
    {"salary": 1500, "historico": 1, "aprovado": 0},
    {"salary": 7000, "historico": 8, "aprovado": 1},
    {"salary": 3000, "historico": 4, "aprovado": 0}
])

X = df[["salary", "historico"]]
y = df["aprovado"]

scaler = StandardScaler() # utiliza pra não dar erro de escala
X_scaled = scaler.fit_transform(X)

# para comparação
X_scaled_df = pd.DataFrame(X_scaled, columns=["salary", "historico"])
X_scaled_df["aprovado"] = y.values

# treina o KNN
modelo = KNeighborsClassifier(n_neighbors=3)
modelo.fit(X_scaled, y)

print(f"Modelo treinado com dados normalizados")
print(X_scaled_df.round(2))

Modelo treinado com dados normalizados
   salary  historico  aprovado
0    0.57       0.95         1
1   -1.35      -1.12         0
2    1.05       1.29         1
3   -0.87      -0.77         0
4    0.57       0.60         1
5   -1.11      -1.46         0
6    1.52       0.95         1
7   -0.39      -0.43         0


# Teste com novos clientes e dados normalizados

In [4]:
novos = pd.DataFrame([
    {"salary": 5000, "historico": 9},
    {"salary": 5001, "historico": 1}
])

novo_scaled = scaler.transform(novos)
previsoes = modelo.predict(novo_scaled)

for i, prev in enumerate(previsoes):
    label = "Aprovado" if prev == 1 else "Reprovado"
    print(f"Cliente {i+1}: {label}")

Cliente 1: Aprovado
Cliente 2: Reprovado


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_cls = df[["salary", "historico"]]
y_cls = df["aprovado"]

# test_size=0.25 para quando for separ a qtd de testes não pegar só os com aprovado = true, ou seja, mantém a proporção das classea aprovado e reprovado
X_train, X_test, y_train, y_test = train_test_split(
    X_cls, y_cls, test_size=0.25, random_state=42, stratify=y_cls
) 


print("Retorno do train_test_split:")
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"y_train: {y_train.shape} | y_test: {y_test.shape}")
print()

# escala com dados de treino e aplica no teste
scaler_cls = StandardScaler()
X_train_s = scaler_cls.fit_transform(X_train)
X_test_s = scaler_cls.transform(X_test)

# treinar e avaliar 
modelo_cls = KNeighborsClassifier(n_neighbors=3)
modelo_cls.fit(X_train, y_train)
y_prev = modelo_cls.predict(X_test)

print(classification_report(
    y_test,
    y_prev,
    labels=[0,1],
    target_names=["Reprovado", "Aprovado"]
))

Retorno do train_test_split:
X_train: (6, 2) | X_test: (2, 2)
y_train: (6,) | y_test: (2,)

              precision    recall  f1-score   support

   Reprovado       1.00      1.00      1.00         1
    Aprovado       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2

